# KALIA v0.3.0 Step 0 — CL-0 forgetting baseline

Prerequisite for every later step: the retention ledger they are scored
against. `tools/forgetting_probe.py` had never been run before this; its
bits-per-byte divided by a hardcoded constant that was ~30% off our corpus,
now measured and recorded.

**Registered expectation:** the probe will show measurable forgetting, because
that is what continual updates do. If it shows none, the probe is broken and we
stop and fix the probe — the ledger is the instrument, and a broken instrument
makes every later number meaningless.

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402, 1403]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    assert hits, 'attach the kalia-code-dev dataset'
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
# Needs the v0.2.0 checkpoint and a frozen probe shard. Both are published
# to the hub, so fetch by range rather than downloading a full snapshot --
# the user is on metered mobile data and a 334MB pull is not acceptable.
ckpt_hits = sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True))
if ckpt_hits:
    ckpt = ckpt_hits[0]
    print('checkpoint: attached dataset file (no hub pull needed)')
else:
    from huggingface_hub import hf_hub_download
    REPO = 'kalia-lm/kalia-v020'
    ckpt = hf_hub_download(REPO, 'checkpoints/ckpt.pt', repo_type='model')
size_mb = os.path.getsize(ckpt) / 1e6
print(f'checkpoint: {ckpt} ({size_mb:.1f} MB)')
assert size_mb > 10, 'checkpoint looks truncated'

probe_bin = None
for cand in ('corpus/probe_val.bin',):
    try:
        probe_bin = hf_hub_download(REPO, cand, repo_type='model')
        print('probe shard:', cand)
        break
    except Exception as e:
        print(f'  {cand}: {type(e).__name__}')
assert probe_bin, (
    'no published probe shard. Run kalia-publish-sources first. Without a\n'
    'frozen probe slice the CL-0 ledger has no fixed measuring set, and\n'
    'drift between updates would be compared against different data each\n'
    'time. Substituting val.bin silently would keep the numbers coming\n'
    'while making the ledger incomparable across updates.')

In [ ]:
ledger = '/kaggle/working/cl0_ledger.jsonl'
r = subprocess.run(
    [sys.executable, 'tools/forgetting_probe.py',
     '--ckpt', ckpt, '--probe', probe_bin, '--ledger', ledger,
     '--batches', '25', '--batch-size', '8', '--device', 'cuda',
     '--commit', 'v0.2.0 baseline'],
    capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode:
    print('STDERR:', r.stderr[-2000:])
rows = [json.loads(l) for l in open(ledger) if l.strip()] if os.path.exists(ledger) else []
print(f'\nledger rows: {len(rows)}')
assert rows, 'CL-0 produced no ledger row -- the probe is broken, stop and fix it'
rec = rows[-1]
print('probe_loss', rec['probe_loss'], '| bpB', rec['bpB'],
      '| bytes/token', rec['bytes_per_token'])
json.dump(rec, open('/kaggle/working/cl0_baseline.json', 'w'), indent=2)

In [ ]:
print('=== Step 0 verdict, fixed in advance ===')
rec = json.load(open('/kaggle/working/cl0_baseline.json'))
# The registered pass condition is a *measurable* baseline, i.e. a finite,
# positive loss. There is no threshold to game here; the point is that the
# instrument works before it is used to judge anything.
ok = rec['probe_loss'] > 0 and rec['probe_tokens'] > 0
print('  CL-0 baseline recorded:', ok)
print(f"  v0.2.0 held-out probe loss {rec['probe_loss']:.4f}"
      f" ({rec['bpB']} bpB)")
print('  Next: Step 1 measures the 58M control seed spread. Every')
print('  threshold in the pre-registration is expressed as 1x that number,')
print('  so it is not yet knowable. That is deliberate.')